In [44]:
import pandas as pd
import numpy as np
eqcsv = pd.read_csv('earthquakes.csv')
faangcsv = pd.read_csv('faang.csv')


In [9]:
print(eqcsv.dtypes)

mag             float64
magType             str
time              int64
place               str
tsunami           int64
parsed_place        str
dtype: object


In [11]:
eqcsv.head()

,mag,magType,time,place,tsunami,parsed_place
0,1.35,ml,1539475168010,"9km NE of Aguanga, CA",0,California
1,1.29,ml,1539475129610,"9km NE of Aguanga, CA",0,California
2,3.42,ml,1539475062610,"8km NE of Aguanga, CA",0,California
3,0.44,ml,1539474978070,"9km NE of Aguanga, CA",0,California
4,2.16,md,1539474716050,"10km NW of Avenal, CA",0,California


In [14]:
#With the earthquakes.csv file, select all the earthquakes 
#in Japan with a magType of mb and a magnitude of 4.9 or greater.

selected = eqcsv[(eqcsv['magType'] == "mb") & (eqcsv['mag'] >= 4.9)]
print(selected)

      mag magType           time                                     place  \
227   5.2      mb  1539389603790                   15km WSW of Pisco, Peru   
229   4.9      mb  1539389546300              193km N of Qulansiyah, Yemen   
248   4.9      mb  1539382925190        151km S of Severo-Kuril'sk, Russia   
258   5.1      mb  1539380306940             236km NNW of Kuril'sk, Russia   
391   5.1      mb  1539337221080                   Pacific-Antarctic Ridge   
...   ...     ...            ...                                       ...   
9154  4.9      mb  1537268270010                    Southwest Indian Ridge   
9175  5.2      mb  1537262729590               126km N of Dili, East Timor   
9176  5.2      mb  1537262656830       90km S of Raoul Island, New Zealand   
9213  5.1      mb  1537255481060                            South of Tonga   
9304  5.1      mb  1537236235470  34km NW of Finschhafen, Papua New Guinea   

      tsunami             parsed_place  
227         0         

In [23]:
#Create bins for each full number of magnitude 
# (for example, the first bin is 0-1, the second is 1-2, and so on) 
# with a magType of ml and count how many are in each bin.
mldata = eqcsv[eqcsv['magType'] == 'ml']
max_mag = int(np.ceil(mldata['mag'].max()))
bins = np.arange(0, max_mag + 1, 1)
mldata['mag_bin'] = pd.cut(mldata['mag'], bins=bins)

bin_counts = mldata['mag_bin'].value_counts().sort_index()
print(bin_counts)

mag_bin
(0, 1]    2207
(1, 2]    3105
(2, 3]     862
(3, 4]     122
(4, 5]       2
(5, 6]       1
Name: count, dtype: int64


In [29]:
print(faangcsv.dtypes)

ticker        str
date          str
open      float64
high      float64
low       float64
close     float64
volume      int64
dtype: object


In [34]:
#Using the faang.csv file, group by the ticker and resample
# to monthly frequency. Make the following aggregations:
grpd = faangcsv.groupby('ticker')

#Mean of the opening price
opMean = grpd['open'].mean()
print(opMean)



ticker
AAPL     187.038674
AMZN    1644.072669
FB       171.454424
GOOG    1113.554104
NFLX     319.620533
Name: open, dtype: float64


In [36]:
#Maximum of the high price
maxHigh = grpd['high'].max()
print(maxHigh)

ticker
AAPL     231.6645
AMZN    2050.5000
FB       218.6200
GOOG    1273.8900
NFLX     423.2056
Name: high, dtype: float64


In [37]:
#Minimum of the low price
minLow = grpd['low'].min()
print(minLow)

ticker
AAPL     145.9639
AMZN    1170.5100
FB       123.0200
GOOG     970.1100
NFLX     195.4200
Name: low, dtype: float64


In [38]:
#Mean of the closing price
aveClose = grpd['close'].mean()
print(aveClose)

ticker
AAPL     186.986218
AMZN    1641.726175
FB       171.510936
GOOG    1113.225139
NFLX     319.290299
Name: close, dtype: float64


In [39]:
#Sum of the volume traded
sumVol  = grpd['volume'].sum()
print(sumVol)

ticker
AAPL    8539383858
AMZN    1418040266
FB      6949682394
GOOG     437403914
NFLX    2879045091
Name: volume, dtype: int64


In [48]:
#Build a crosstab with the earthquake data between the tsunami
#column and the magType column. Rather than showing the 
#frequency count, show the maximum magnitude that was observed 
#for each combination. Put the magType along the columns.
xtab = pd.crosstab(eqcsv['tsunami'], eqcsv['magType'])
print(xtab)


magType   mb  mb_lg    md  mh    ml  ms_20  mw  mwb  mwr  mww
tsunami                                                      
0        574     30  1796  12  6798      0   2    2   14   42
1         27      0     0   0     5      1   2    0    0   26


In [58]:
# Calculate the rolling 60-day aggregations of OHLC data by 
# ticker for the FAANG data. Use the same aggregations as 
# exercise no. 3.

faangcsv['date'] = pd.to_datetime(faangcsv['date'])
faang = faangcsv.sort_values(['ticker', 'date'])
faang = faang.set_index('date')

rolling_60days = (faang
                  .groupby('ticker')
                  .rolling('60D')
                  .agg({
                      'open': 'first',
                      'high': 'max',
                      'low': 'min',
                      'close': 'last'
                  })
                  .reset_index()
)

rolling_60days

,ticker,date,open,high,low,close
0,AAPL,2018-01-02,166.9271,169.0264,166.0442,168.9872
1,AAPL,2018-01-03,166.9271,171.2337,166.0442,168.9578
2,AAPL,2018-01-04,166.9271,171.2337,166.0442,169.7426
3,AAPL,2018-01-05,166.9271,172.0381,166.0442,171.6751
4,AAPL,2018-01-08,166.9271,172.2736,166.0442,171.0375
...,...,...,...,...,...,...
1250,NFLX,2018-12-24,300.5100,332.0499,233.6800,233.8800
1251,NFLX,2018-12-26,305.2600,332.0499,231.2300,253.6700
1252,NFLX,2018-12-27,305.2600,332.0499,231.2300,255.5650
1253,NFLX,2018-12-28,275.5700,332.0499,231.2300,256.0800


In [59]:
print(faangcsv.dtypes)

ticker               str
date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume             int64
dtype: object


In [ ]:
#Create a pivot table of the FAANG data that compares the 
#stocks. Put the ticker in the rows and show the averages
#of the OHLC and volume traded data.\

pivTable = pd.pivot_table(faangcsv, values =  '', index = '', aggfunc = 'mean')


In [ ]:
# Calculate the Z-scores for each numeric column of Netflix's 
# data (ticker is NFLX) using apply().


In [ ]:
#Create a dataframe with the following three columns: ticker, date, and event. The columns should have the following values:
#ticker: 'FB'
#date: ['2018-07-25', '2018-03-19', '2018-03-20']
#event: ['Disappointing user growth announced after close.', 'Cambridge Analytica story', 'FTC investigation']
#Set the index to ['date', 'ticker']